### **Results and Community Profiling**

The script iterates through the communities identified in the previous phases (specifically the Top Communities within the Giant Component) to generate a structured "Identity Card" for each social group.

The analysis is performed by the custom function `generate_community_profile`, which extracts four dimensions of analysis:

1.  **Key Statistics & Sentiment:**
    * Calculates group size and engagement (comments per user).
    * Displays the **BERT Sentiment** to determine the group's political role (Supporter vs. Opponent).

2.  **Thematic Identity:**
    * Retrieves the **Semantic Keyword** (the "core concept" calculated via SBERT).
    * Lists the **Top 3 Topics** discussed within the specific community to understand its unique focus.

3.  **Leadership Analysis (Popularity vs. Structure):**
    * **Popularity Leader:** The user with the highest aggregate Karma (Vote Score).
    * **Structural Leader:** The user with the highest **Betweenness Centrality** (derived from the SNA phase). This comparison allows us to see if the most "liked" users are the same ones controlling the information flow.

4.  **Qualitative Context:**
    * Extracts and prints the top 3 comments by score to provide immediate textual context to the numerical data.

The output is printed to the console for immediate review and saved as a persistent text file (`chat_control_FINAL_RESULTS.txt`).

In [ ]:
import pandas as pd
import os
from google.colab import drive

print("--- GENERATING DETAILED TEXTUAL REPORT (Console Output) ---")

# --- 1. CONFIGURATION ---
INPUT_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Datasets"
OUTPUT_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Results"
INPUT_FILE = "chat_control_FINAL.csv"
OUTPUT_REPORT = "chat_control_FINAL_RESULTS.txt"

path_in = os.path.join(INPUT_PATH, INPUT_FILE)
path_out = os.path.join(OUTPUT_PATH, OUTPUT_REPORT)

# Mount Google Drive
try:
    drive.mount('/content/drive', force_remount=True)
except:
    pass

# Validation
if not os.path.exists(path_in):
    raise SystemExit(f"ERROR: File {path_in} not found.")

df = pd.read_csv(path_in)
print(f"[DATA] Loaded {len(df)} comments.")

# --- 2. PROFILING FUNCTION ---
def generate_community_profile(df_comm, group_name):
    profile = []
    separator = "=" * 50
    profile.append(separator)
    profile.append(f"REPORT: {group_name.upper()}")
    profile.append(separator)

    # A. Base Statistics
    n_comments = len(df_comm)
    n_users = df_comm['comment_author'].nunique()

    # Sentiment
    avg_vader = df_comm['vader_score'].mean()
    avg_bert = df_comm['bert_score'].mean()

    # Hypothesized Role based on BERT
    role = "Neutral"
    if avg_bert > 0.05: role = "SUPPORTER (Positive)"
    elif avg_bert < -0.05: role = "OPPONENT (Negative)"

    profile.append(f"\n[1] KEY STATISTICS:")
    profile.append(f"- Size: {n_users} users ({n_comments} comments)")

    activity = n_comments/n_users if n_users > 0 else 0
    profile.append(f"- Avg Activity: {activity:.2f} comments/user")
    profile.append(f"- Sentiment BERT: {avg_bert:.4f} -> {role}")
    profile.append(f"- Sentiment VADER: {avg_vader:.4f}")

    # B. Thematic Analysis (Semantic Keyword & Topic)
    # Get the most frequent keyword or the first available
    sem_key = df_comm['semantic_keyword'].iloc[0] if 'semantic_keyword' in df_comm.columns else "N/A"

    # Calculate internal Topic distribution (Top 3)
    top_topics = df_comm['topic_name'].value_counts().head(3)

    profile.append(f"\n[2] THEMATIC IDENTITY:")
    profile.append(f"- Semantic Keyword (Core Concept): {sem_key}")
    profile.append(f"- Top 3 Topics Discussed: ")
    if n_comments > 0:
        for topic, count in top_topics.items():
            pct = (count / n_comments) * 100
            profile.append(f"   * {topic}: {pct:.1f}%")
    else:
        profile.append("   * No topics detected")

    # C. Top Influencers (Comparison: Popularity vs Structure)
    profile.append(f"\n[3] OPINION LEADERS:")

    # 1. Popularity Leader (Highest Reddit Score/Karma)
    top_karma = df_comm.groupby('comment_author')['comment_score'].sum().sort_values(ascending=False).head(1)
    if not top_karma.empty:
        profile.append(f"   - Popularity Leader (Max Score): {top_karma.index[0]} (Score: {top_karma.iloc[0]})")

    # 2. Structural Leader (Highest Betweenness - from SNA)
    if 'betweenness_centrality' in df_comm.columns:
        top_struct = df_comm.sort_values('betweenness_centrality', ascending=False).head(1)
        if not top_struct.empty:
            auth = top_struct.iloc[0]['comment_author']
            val = top_struct.iloc[0]['betweenness_centrality']
            profile.append(f"   - Structural Leader (Max Betweenness): {auth} (Val: {val:.4f})")

    # D. Representative Comments (Highest Score)
    top_comments = df_comm.sort_values('comment_score', ascending=False).head(3)

    profile.append(f"\n[4] TOP RATED COMMENTS:")
    for i, (idx, row) in enumerate(top_comments.iterrows(), 1):
        profile.append(f"\n   #{i} [Author: {row['comment_author']} | Score: {row['comment_score']}]")
        # Truncate text for readability
        text_preview = str(row['comment_body']).replace('\n', ' ')[:300]
        profile.append(f"   \"{text_preview}...\"")

    profile.append("\n\n")
    return "\n".join(profile)

# --- 3. GENERATION AND EXPORT ---
print("\n[REPORT] Starting profile generation...\n")

full_report = "IN-DEPTH COMMUNITY ANALYSIS - CHAT CONTROL\n\n"

# Validate Data
if 'community_group' not in df.columns:
     raise ValueError("Column 'community_group' not found. Ensure Phase 3 (SCA) was run.")

# Identify target groups (Top Communities ONLY - No Isolates in Giant Component)
target_groups = [g for g in df['community_group'].unique() if "Comm_" in g]

# Sort by size to print largest communities first
group_sizes = df['community_group'].value_counts()
sorted_groups = [g for g in group_sizes.index if g in target_groups]

for group_name in sorted_groups:
    # Filter data
    subset = df[df['community_group'] == group_name]

    # Generate profile
    report_text = generate_community_profile(subset, group_name)

    # Print to Console
    print(report_text)
    print("-" * 20)

    # Append to full report string
    full_report += report_text

# Save to file
with open(path_out, "w", encoding="utf-8") as f:
    f.write(full_report)

print(f"\nDONE. Full report saved to: {path_out}")

### **Visualizations**

In this phase, we translate the numerical metrics into **visual insights** to better understand the dynamics of the "Chat Control" debate. The code generates five critical visualizations using `matplotlib`, `seaborn`, and `wordcloud`:

1.  **Thematic Composition (Stacked Bar Chart):**
    * **Objective:** Visualizes the distribution of the top 8 topics within each community.
    * **Insight:** It allows us to distinguish between communities focused on single issues (e.g., "Danish Politics") and those with a broader, multi-thematic agenda.

2.  **Sentiment Polarization (Boxplot):**
    * **Objective:** Displays the distribution of **BERT Sentiment scores** for each group.
    * **Insight:** The "box" shows the coherence of the group. A tight box indicates a unified consensus (echo chamber), while a wide box suggests internal disagreement.

3.  **Temporal Evolution (Time Series):**
    * **Objective:** Tracks the daily average sentiment over time.
    * **Insight:** Identifies specific dates or events that triggered spikes in negativity across the network.

4.  **Topic Toxicity Ranking (Horizontal Bar Chart):**
    * **Objective:** Ranks the most discussed topics based on their average sentiment.
    * **Specific Focus:** The visualization is specifically zoomed in on the **negative spectrum** (from -1 to 0), highlighting which specific themes (e.g., "Surveillance," "Fascism") generate the most hostile reactions.

5.  **Semantic WordClouds:**
    * **Objective:** Visualizes keywords based on **SBERT semantic relevance** (Cosine Similarity) rather than simple frequency, dynamically colored by the group's average sentiment.
    * **Insight:** Filters out noise to reveal the **defining concepts** of each community, providing an immediate snapshot of their specific ideological stance (e.g., distinguishing a focus on "Child Protection" from "Anti-Surveillance").

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import numpy as np

print("--- VISUALIZATION ---")

# --- CONFIGURATION ---
OUTPUT_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Results"
IMG_PATH = os.path.join(OUTPUT_PATH, "IMAGES")
os.makedirs(IMG_PATH, exist_ok=True)

# Filter relevant communities (Top Communities only)
target_groups = [g for g in df['community_group'].unique() if "Comm_" in g]
df_viz = df[df['community_group'].isin(target_groups)].copy()

# Order by community size
order_list = df_viz['community_group'].value_counts().index

# Graphics Style
sns.set_theme(style="whitegrid")

# ==============================================================================
# 1. STACKED BAR CHART: TOPIC COMPOSITION
# ==============================================================================
print("\n[1/4] Generating Topic Composition...")

# Calculate percentages
topic_composition = pd.crosstab(df_viz['community_group'], df_viz['topic_name'], normalize='index') * 100

# Select Top 8 topics to avoid clutter
top_n_topics = 8
top_topics_cols = df_viz['topic_name'].value_counts().head(top_n_topics).index.tolist()
viz_data = topic_composition[top_topics_cols].copy()
viz_data['Other Topics'] = 100 - viz_data.sum(axis=1)

# Plot
plt.figure(figsize=(14, 8))
viz_data.loc[order_list].plot(kind='bar', stacked=True, colormap='tab10', figsize=(14, 8), width=0.8)

plt.title("Thematic Composition by Community", fontsize=16)
plt.ylabel("Percentage (%)", fontsize=12)
plt.xlabel("Community", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title="Main Topics", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(IMG_PATH, "viz1_topic_composition.png"), dpi=300)
plt.show()

# ==============================================================================
# 2. BOXPLOT: SENTIMENT DISTRIBUTION
# ==============================================================================
print("\n[2/4] Generating Sentiment Boxplot...")

plt.figure(figsize=(14, 8))
sns.boxplot(x='community_group', y='bert_score', data=df_viz, order=order_list, palette="coolwarm_r")

plt.axhline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
plt.title("Sentiment Distribution and Polarization (BERT)", fontsize=16)
plt.ylabel("Sentiment Score (-1 Negative / +1 Positive)", fontsize=12)
plt.xlabel("Community", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(IMG_PATH, "viz2_sentiment_distribution.png"), dpi=300)
plt.show()

# ==============================================================================
# 3. TIME SERIES: SENTIMENT EVOLUTION
# ==============================================================================
print("\n[3/4] Generating Sentiment Time Series...")

if 'comment_created_utc' in df_viz.columns:
    df_viz['date'] = pd.to_datetime(df_viz['comment_created_utc'], unit='s')
    df_time = df_viz.set_index('date').sort_index()

    plt.figure(figsize=(14, 7))

    # Daily Mean
    daily_sentiment = df_time.resample('D')['bert_score'].mean()

    # Plot Line
    sns.lineplot(data=daily_sentiment, label="Daily Average Sentiment", color="blue", linewidth=2)

    plt.axhline(0, color='black', linestyle='-', linewidth=0.5)
    plt.title("Temporal Evolution of Global Sentiment", fontsize=16)
    plt.ylabel("Average Sentiment", fontsize=12)
    plt.xlabel("Date", fontsize=12)
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(IMG_PATH, "viz3_sentiment_time_series.png"), dpi=300)
    plt.show()
else:
    print("Date column not found.")

# ==============================================================================
# 4. HORIZONTAL BAR CHART: TOPIC RANKING (NEGATIVE FOCUS)
# ==============================================================================
print("\n[4/4] Generating Topic Ranking (Negative Focus)...")

# 1. Calculate mean sentiment per topic
topic_stats = df_viz.groupby('topic_name')['bert_score'].mean().sort_values()

# 2. Select top 12 most frequent topics
top_topics = df_viz['topic_name'].value_counts().head(12).index
df_best_topics = topic_stats[topic_stats.index.isin(top_topics)].sort_values()

# 3. Conditional Colors (Red for Negative, Green for Positive)
colors = ['#d62728' if x < 0 else '#2ca02c' for x in df_best_topics.values]

plt.figure(figsize=(10, 8))

# Horizontal Bar Plot
plt.barh(df_best_topics.index, df_best_topics.values, color=colors, alpha=0.8)

# Focus on Negative Axis
plt.xlim(-1, 0.05)
plt.axvline(0, color='black', linewidth=1)

plt.title("Topic Ranking: Focus on Negativity", fontsize=14, pad=15)
plt.xlabel("Average Sentiment (Scale: -1 Very Negative | 0 Neutral)", fontsize=12)

# Add value labels on bars
for index, value in enumerate(df_best_topics.values):
    label_pos = value - 0.02 if value < 0 else 0.01
    # Adjust alignment based on value
    plt.text(label_pos, index, f"{value:.2f}", va='center', fontsize=10, color='black', ha='right' if value < 0 else 'left')

plt.tight_layout()
plt.savefig(os.path.join(IMG_PATH, "viz4_simple_topics_negative_focus.png"), dpi=300)
plt.show()

print(f"\nCharts saved to: {IMG_PATH}")

In [ ]:
# --- LIBRARY INSTALLATION ---
try:
    from wordcloud import WordCloud, STOPWORDS
except ImportError:
    !pip install -q wordcloud
    from wordcloud import WordCloud, STOPWORDS

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os
import re
import math

print("\n--- GENERATING SEMANTIC WORDCLOUDS (GRID VIEW) ---")

# --- 1. CONFIGURATION ---
INPUT_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Datasets"
OUTPUT_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Results"
IMG_PATH_WC = os.path.join(OUTPUT_PATH, "IMAGES_COMMUNITY_WC")
os.makedirs(IMG_PATH_WC, exist_ok=True)

# Dataset Verification
if 'df' not in locals():
    INPUT_FILE = "chat_control_FINAL.csv"
    path_in = os.path.join(INPUT_PATH, INPUT_FILE)
    if os.path.exists(path_in):
        df = pd.read_csv(path_in)
    else:
        raise SystemExit("Dataset not found.")

# --- 2. UTILITY FUNCTIONS ---

def create_circular_mask(h, w):
    """Creates a circular mask for the WordCloud."""
    center = (int(w/2), int(h/2))
    radius = min(center[0], center[1], w-center[0], h-center[1])
    Y, X = np.ogrid[:h, :w]
    dist_from_center = np.sqrt((X - center[0])**2 + (Y-center[1])**2)
    mask = dist_from_center > radius
    return mask.astype(int) * 255

class GradientColorFunc(object):
    """Generates a color gradient based on word frequency/weight."""
    def __init__(self, word_freqs, hue_start):
        self.word_freqs = word_freqs
        self.hue = hue_start
        self.max_freq = max(word_freqs.values()) if word_freqs else 1

    def __call__(self, word, font_size, position, orientation, random_state=None, **kwargs):
        freq = self.word_freqs.get(word, 0)
        rel_freq = freq / self.max_freq
        lightness = int(85 - (rel_freq * 55))
        return f"hsl({self.hue}, 90%, {lightness}%)"

# --- 3. DATA PREPARATION ---
target_groups = [g for g in df['community_group'].unique() if "Comm_" in g]
df_filtered = df[df['community_group'].isin(target_groups)].copy()

grouped = df_filtered.groupby('community_group')

# --- 4. GRID SETUP ---
# Calculate rows needed for 2 columns
n_groups = len(target_groups)
n_cols = 2
n_rows = math.ceil(n_groups / n_cols)

print(f"Generating WordClouds for {n_groups} communities (Grid: {n_rows}x{n_cols})...")

# Create the main figure for display (Big Grid)
# figsize=(Width, Height) -> We make it tall to accommodate all rows clearly
fig_grid, axes = plt.subplots(n_rows, n_cols, figsize=(20, 10 * n_rows))
axes_flat = axes.flatten() # Flatten to 1D array for easy iteration

mask = create_circular_mask(1000, 1000)

for i, group_name in enumerate(target_groups):

    # Get current axis for the grid
    ax_grid = axes_flat[i]

    # A. Get Group Data
    group_data = grouped.get_group(group_name)
    sentiment = group_data['bert_score'].mean()

    # B. Retrieve and Weight Keywords
    if 'community_all_keywords' not in group_data.columns:
        ax_grid.text(0.5, 0.5, "Data Missing", ha='center')
        continue

    keywords_raw = group_data['community_all_keywords'].dropna().iloc[0]

    if not isinstance(keywords_raw, str) or not keywords_raw.strip():
        ax_grid.text(0.5, 0.5, "No Keywords", ha='center')
        continue

    keywords_list = [k.strip() for k in keywords_raw.split(',')]

    # Calculate Positional Weights
    total_kws = len(keywords_list)
    frequencies = {}
    for j, word in enumerate(keywords_list):
        frequencies[word] = total_kws - j

    # C. Color Configuration
    if sentiment < -0.05:
        base_hue = 0        # Red (Negative)
        border_color = "#d62728"
        label = "Negative"
    elif sentiment > 0.05:
        base_hue = 120      # Green (Positive)
        border_color = "#2ca02c"
        label = "Positive"
    else:
        base_hue = 210      # Blue (Neutral)
        border_color = "#1f77b4"
        label = "Neutral"

    try:
        # D. Generate WordCloud Object
        wc = WordCloud(
            background_color="white",
            width=1000, height=1000,
            mask=mask,
            max_words=total_kws,
            max_font_size=250, min_font_size=10,
            random_state=42,
            prefer_horizontal=0.9,
            contour_width=3,
            contour_color=border_color,
            relative_scaling=0.5
        ).generate_from_frequencies(frequencies)

        gradient_func = GradientColorFunc(frequencies, base_hue)
        wc.recolor(color_func=gradient_func)

        # --- ACTION 1: PLOT TO GRID (DISPLAY) ---
        ax_grid.imshow(wc, interpolation="bilinear")
        ax_grid.axis("off")
        ax_grid.set_title(f"{group_name}\n({label}: {sentiment:.2f})", fontsize=18, color=border_color, fontweight='bold', pad=15)

        # --- ACTION 2: SAVE TO DRIVE (INDIVIDUAL FILE) ---
        # We create a temporary figure to save the single file cleanly without affecting the grid
        temp_fig = plt.figure(figsize=(10, 10))
        plt.imshow(wc, interpolation="bilinear")
        plt.axis("off")
        plt.title(f"{group_name}\n({label}: {sentiment:.2f})", fontsize=24, color=border_color, fontweight='bold', pad=20)

        filename = f"{group_name}_wordcloud_keywords.png"
        save_path = os.path.join(IMG_PATH_WC, filename)
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.close(temp_fig) # Close temp figure immediately

        print(f" -> Processed: {group_name}")

    except ValueError as e:
        print(f"Error generating {group_name}: {e}")
        ax_grid.text(0.5, 0.5, "Error", ha='center')

# Hide empty subplots if total groups are odd (e.g., 9 groups in a 10-slot grid)
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].axis('off')
    axes_flat[j].set_visible(False)

print(f"\nDONE. Individual images saved to: {IMG_PATH_WC}")
print("Displaying Grid Overview below:")

plt.tight_layout()
plt.show()

### **Interactive Visualizations**

To complement the static analysis, we developed two **interactive dashboards** using `Plotly`. These tools allow for a qualitative "drill-down" into the data, enabling the exploration of specific data points and hierarchical relationships that static charts cannot convey.

1.  **Opinion Explorer (Interactive Bubble Chart):**
    * **Objective:** Maps the relationship between **Sentiment** (X-axis) and **Popularity** (Y-axis), where the bubble size represents the comment's score.
    * **Insight:** Unlike static scatterplots, this tool allows for **granular inspection**. By hovering over individual bubbles, we can read the raw text, identify the author, and see the specific topic. This is crucial for qualitative validation: it helps us understand *why* a specific negative comment went viral or *what* a controversial outlier is actually saying.

2.  **Debate Structure (Sunburst Chart):**
    * **Objective:** Visualizes the hierarchical relationship between **Communities** (Inner Ring) and their **Dominant Topics** (Outer Ring).
    * **Insight:** This chart reveals the "Topic Ownership." By interacting with the segments, we can instantly assess the thematic diversity of each group. It visually answers: *Is Community X mono-thematic (obsessed with one issue) or does it engage in a broader, multi-faceted debate?*

In [ ]:
import plotly.express as px
import os
import pandas as pd

print("--- CHART 1: INTERACTIVE SCATTERPLOT ---")

# Configuration
IMG_PATH = os.path.join(OUTPUT_PATH, "IMAGES_INTERACTIVE")
os.makedirs(IMG_PATH, exist_ok=True)

# 1. Data Preparation
# Filter: Keep only Top Communities (No Isolates)
target_groups = [g for g in df['community_group'].unique() if "Comm_" in g]
df_scatter = df[df['community_group'].isin(target_groups)].copy()

# FIX 1: Handle NaN in text column
df_scatter['clean_text'] = df_scatter['clean_text'].fillna("").astype(str)

# Create Text Preview for Tooltip
df_scatter['text_preview'] = df_scatter['clean_text'].apply(lambda x: x[:120] + "..." if len(x) > 120 else x)

# FIX 2: Negative Size Handling
# Plotly crashes if size < 0. We create a visual metric: min size 2, otherwise the score value.
df_scatter['visual_size'] = df_scatter['comment_score'].apply(lambda x: max(x, 2))

# Noise Filter (Optional)
# We filter out low-value points (low score AND neutral sentiment) to improve performance/clarity
mask_relevant = (df_scatter['comment_score'] > 2) | (df_scatter['bert_score'].abs() > 0.1)
df_scatter_clean = df_scatter[mask_relevant]

# 2. Chart Creation
fig = px.scatter(
    df_scatter_clean,
    x="bert_score",
    y="comment_score",
    color="community_group",

    # Use the corrected positive variable for bubble size
    size="visual_size",
    size_max=40,

    hover_name="community_group",
    hover_data={
        "topic_name": True,
        "comment_author": True,
        "text_preview": True,
        "bert_score": False,   # Already on axis
        "comment_score": True, # Show real score (even if negative) in tooltip
        "community_group": False,
        "visual_size": False   # Hide technical variable
    },
    title="<b>Opinion Explorer:</b> Sentiment vs. Popularity (Hover for details)",
    labels={
        "bert_score": "Sentiment (Negative ⬅ ➡ Positive)",
        "comment_score": "Upvotes (Popularity)",
        "community_group": "Community"
    },
    template="plotly_white",
    opacity=0.7
)

# Add Zero Line for Sentiment
fig.add_vline(x=0, line_width=1, line_dash="dash", line_color="black")

# 3. Save and Show
outfile = os.path.join(IMG_PATH, "interactive_1_scatter.html")
fig.write_html(outfile)
print(f"Interactive chart saved to: {outfile}")
fig.show()

In [ ]:
import plotly.express as px
import os

print("--- CHART 2: SUNBURST CHART ---")

# Configuration
IMG_PATH = os.path.join(OUTPUT_PATH, "IMAGES_INTERACTIVE")
os.makedirs(IMG_PATH, exist_ok=True)

# 1. Data Preparation
# Aggregate: Count comments for each Community -> Topic pair
df_sunburst = df.groupby(['community_group', 'topic_name']).size().reset_index(name='counts')

# Filter: Keep only Top Communities (SNA Giant Component)
target_groups = [g for g in df['community_group'].unique() if "Comm_" in g]
df_sunburst = df_sunburst[df_sunburst['community_group'].isin(target_groups)]

# Filter: Remove micro-topics (<15 comments) for visual clarity
df_sunburst = df_sunburst[df_sunburst['counts'] >= 15]

# 2. Chart Creation
fig = px.sunburst(
    df_sunburst,
    path=['community_group', 'topic_name'], # Hierarchy: Inner -> Outer
    values='counts',                        # Slice size
    color='community_group',                # Color based on parent Community
    title="<b>Debate Structure:</b> Communities and their Dominant Topics",
    template="plotly_white"
)

# Enhance Tooltip
fig.update_traces(
    textinfo="label+percent entry", # Show label and % relative to parent
    hovertemplate='<b>%{label}</b><br>Comments: %{value}<br>Share: %{percentEntry:.1%}'
)

# 3. Save and Show
outfile = os.path.join(IMG_PATH, "interactive_2_sunburst.html")
fig.write_html(outfile)
print(f"Interactive chart saved to: {outfile}")
fig.show()

### **Gephi Export**

This script bridges quantitative analysis and visual exploration by constructing the final network graph, where nodes represent unique users and edges represent their interactions (replies).

Each node is richly attributed with data from previous phases to enable multi-dimensional filtering in Gephi:
* **Social Metrics:** Activity, Popularity (Total Score), and average Sentiment.
* **Structural Metrics:** Degree and Betweenness Centrality (imported from the SNA phase).
* **Thematic Identity:** Community membership, dominant topics, and semantic keywords.

The final output is a `.gexf` file (`chat_control_network.gexf`), ready for advanced layout algorithms and visual analysis.

In [ ]:
import pandas as pd
import networkx as nx
import os
from tqdm.auto import tqdm
from google.colab import drive

print("--- PHASE 4: VISUALIZATION & GEPHI EXPORT (With SNA Metrics) ---")

# --- 1. CONFIGURATION ---
INPUT_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Datasets"
OUTPUT_PATH = "/content/drive/MyDrive/MAGISTRALE/Social_Media/SMA_Borgia_Borserini/Results"
INPUT_FILE = "chat_control_FINAL.csv"
OUTPUT_GEXF = "chat_control_network.gexf"

INPUT_PATH = os.path.join(INPUT_PATH, INPUT_FILE)
OUTPUT_PATH = os.path.join(OUTPUT_PATH, OUTPUT_GEXF)

# Mount Drive
try:
    drive.mount('/content/drive', force_remount=True)
except:
    pass

if not os.path.exists(INPUT_PATH):
    raise SystemExit(f"ERROR: File {INPUT_PATH} not found.")

df = pd.read_csv(INPUT_PATH)
print(f"[DATA] Loaded {len(df)} total comments.")

# --- 2. DATASET FILTERING (No Minor Groups) ---
# We keep only the relevant communities for the visualization
df_filtered = df[df['community_group'] != 'Minor_Groups'].copy()
print(f"[FILTER] Comments retained: {len(df_filtered)}")

# --- 3. DATA AGGREGATION ---
print("\n[PREPARATION] Calculating node attributes...")

# A. COMMUNITY Metrics (Attributes to attach to each user based on their group)
comm_stats = df_filtered.groupby('community').agg({
    'bert_score': 'mean',                                          # Avg Sentiment of the group
    'topic_name': lambda x: x.mode()[0] if not x.mode().empty else "Mix", # Dominant Topic
    'semantic_keyword': 'first',                                   # Semantic Label
    'comment_author': 'nunique'
}).to_dict('index')

# B. USER Metrics
# We aggregate comment-level data to user-level nodes
agg_rules = {
    'community': lambda x: x.mode()[0] if not x.mode().empty else -1, # Main community of the user
    'community_group': 'first',
    'bert_score': 'mean',      # User's average sentiment
    'comment_score': 'sum',    # User's total karma (popularity)
    'comment_id': 'count'      # User's activity (number of comments)
}

# Add SNA Centrality metrics if they exist in the dataset
if 'degree_centrality' in df_filtered.columns:
    agg_rules['degree_centrality'] = 'max'
if 'betweenness_centrality' in df_filtered.columns:
    agg_rules['betweenness_centrality'] = 'max'

# Perform Aggregation
df_nodes = df_filtered.groupby('comment_author').agg(agg_rules).reset_index()

# Rename for clarity
df_nodes.rename(columns={'comment_score': 'total_score', 'comment_id': 'num_comments'}, inplace=True)

# Define User Role based on Sentiment
def get_user_role(score):
    if score > 0.05: return "Supporter/Positive"
    if score < -0.05: return "Opponent/Negative"
    return "Neutral"

df_nodes['user_role'] = df_nodes['bert_score'].apply(get_user_role)

print(f"Unique User Nodes: {len(df_nodes)}")

# --- 4. DIRECTED GRAPH CONSTRUCTION (DiGraph) ---
print("\n[GRAPH] Building Directed Network...")
G = nx.DiGraph() # Directed Graph: Preserves A -> B relationship

id_to_author = pd.Series(df_filtered.comment_author.values, index=df_filtered.comment_id).to_dict()
valid_authors_set = set(df_nodes['comment_author'])

# A. Add Nodes with Attributes
for _, row in tqdm(df_nodes.iterrows(), total=len(df_nodes), desc="Adding Nodes"):
    node_id = str(row['comment_author'])
    comm_id = row['community']
    stats = comm_stats.get(comm_id, {})

    # Retrieve SNA values (default 0.0 if missing)
    deg_cent = float(row['degree_centrality']) if 'degree_centrality' in row else 0.0
    bet_cent = float(row['betweenness_centrality']) if 'betweenness_centrality' in row else 0.0

    G.add_node(
        node_id,
        Label=node_id,
        # User Attributes
        User_Sentiment=float(row['bert_score']),
        User_Role=str(row['user_role']),
        User_Score=int(row['total_score']),
        User_Activity=int(row['num_comments']),

        # SNA Metrics (Calculated in Phase 2)
        SNA_Degree_Centrality=deg_cent,
        SNA_Betweenness_Centrality=bet_cent,

        # Community Attributes (Useful for coloring in Gephi)
        Community_ID=int(comm_id),
        Community_Group=str(row['community_group']),
        Community_Avg_Sentiment=float(stats.get('bert_score', 0)),
        Community_Topic=str(stats.get('topic_name', 'Unknown')),
        Community_Semantic_Keyword=str(stats.get('semantic_keyword', 'N/A'))
    )

# B. Add Directed Edges (Replies)
for _, row in tqdm(df_filtered.iterrows(), total=len(df_filtered), desc="Adding Edges"):
    source = row['comment_author']        # AUTHOR (Who is speaking)
    parent_id = str(row['comment_parent_id'])

    if parent_id.startswith('t1_'):       # It's a reply to another comment
        parent_clean = parent_id.split('_')[-1]
        target = id_to_author.get(parent_clean) # TARGET (Who is being replied to)

        if target and (target in valid_authors_set) and (source != target):
            # Add or update edge weight
            if G.has_edge(source, target):
                G[source][target]['weight'] += 1
            else:
                G.add_edge(source, target, weight=1)

print(f"\nFinal Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges.")

# --- 5. EXPORT ---
nx.write_gexf(G, OUTPUT_PATH)
print(f"\nDONE. Gephi file saved to: {OUTPUT_PATH}")